# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: The Content Performance Curve. Content peaks at 61-90 days, declines after 270 days, and the 365+ rebound is concentrated in older pages that were refreshed.**
* Methodology question: Does the decline after 270 days reflect a true decay in relevance, or is it confounded by seasonality or the fact that older content naturally accumulates more outdated facts?
* Why the question matters: If the decline is driven by outdated facts rather than pure age, simple fact-checking updates might reverse the decay without requiring a full structural rewrite.

**Finding 2: The Freshness Multiplier. Content that is 365+ days old refreshed within 30 days shows a 3.2x health boost and 57x more impressions.**
* Methodology question: Were the refreshed pages selected randomly, or did editors prioritize historically strong pages that were already showing signs of life (survivorship/selection bias)?
* Why the question matters: If editors only refreshed proven winners, applying this refresh strategy to historically weak content might not yield the same 57x impression boost.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

def prepare_features(data):
    d = data.copy()
    d["has_word_count"] = d["word_count"].notna().astype(int)
    d["word_count"] = d["word_count"].fillna(0)
    d["has_position_data"] = (d["avg_position"] > 0).astype(int)
    d["log_impressions_90d"] = np.log1p(d["impressions_90d"])
    d["log_clicks_90d"] = np.log1p(d["clicks_90d"])
    return d

df_prep = prepare_features(df)
features = [
    "days_since_last_update", 
    "log_impressions_90d",
    "log_clicks_90d",
    "avg_position", 
    "has_position_data",
    "ctr",
    "word_count",
    "has_word_count",
    "content_age_days"
]
target = "is_declining"

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Strategy 1: Before condition - Actual Week 5 Validation Strategy (Grouped Split by client_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_prep, groups=df_prep["client_id"]))
X_train_grp = df_prep.iloc[train_idx][features]
y_train_grp = df_prep.iloc[train_idx][target]
X_test_grp = df_prep.iloc[test_idx][features]
y_test_grp = df_prep.iloc[test_idx][target]

scaler_grp = StandardScaler()
lr_grp = LogisticRegression(random_state=42, max_iter=1000)
lr_grp.fit(scaler_grp.fit_transform(X_train_grp), y_train_grp)
probs_grp = lr_grp.predict_proba(scaler_grp.transform(X_test_grp))[:, 1]
roc_grp = roc_auc_score(y_test_grp, probs_grp)
p50_grp = precision_at_k(probs_grp, y_test_grp, 50)

# Strategy 2: After condition - Improved Time-Aware Validation (Train on older, Test on newer)
df_time = df_prep.sort_values("content_age_days", ascending=False).reset_index(drop=True)
split_idx = int(len(df_time) * 0.8)
df_train_time = df_time.iloc[:split_idx]
df_test_time = df_time.iloc[split_idx:]

X_train_time = df_train_time[features]
y_train_time = df_train_time[target]
X_test_time = df_test_time[features]
y_test_time = df_test_time[target]

scaler_time = StandardScaler()
lr_time = LogisticRegression(random_state=42, max_iter=1000)
lr_time.fit(scaler_time.fit_transform(X_train_time), y_train_time)
probs_time = lr_time.predict_proba(scaler_time.transform(X_test_time))[:, 1]
roc_time = roc_auc_score(y_test_time, probs_time)
p50_time = precision_at_k(probs_time, y_test_time, 50)

print("### Validation Strategy Comparison")
print("| Validation Strategy | ROC-AUC | Precision@50 |")
print("|---|---|---|")
print(f"| Before: Actual Week 5 (Grouped by `client_id`) | {roc_grp:.3f} | {p50_grp:.3f} |")
print(f"| After: Improved Time-Aware (Train past, Test future) | {roc_time:.3f} | {p50_time:.3f} |")


### Validation Strategy Comparison
| Validation Strategy | ROC-AUC | Precision@50 |
|---|---|---|
| Before: Actual Week 5 (Grouped by `client_id`) | 0.636 | 0.820 |
| After: Improved Time-Aware (Train past, Test future) | 0.678 | 0.880 |


**Why time-aware validation is improved:**
While grouped validation (`client_id`) prevents memorizing client-specific quirks, it still allows the model to train on recent content and predict on older content. A time-aware split (training on the older 80% and testing on the newest 20%) mimics the actual deployment environment where we train on the past to predict the future. This catches temporal leakage and provides an honest estimate for trend-like tasks.

## 3. Leakage audit

| Feature | Keep/Remove | Reason |
|---|---|---|
| client_id | Remove | ID column; prevents generalizing to new clients and causes target leakage if memorized. |
| content_id | Remove | ID column; unique identifier that leads to memorization. |
| trend_direction | Remove | This is used to derive the target label. Absolute target leakage. |
| trend_pct | Remove | This directly correlates with and derives the target label. Absolute target leakage. |
| impressions_90d | Keep | Trailing 90-day metric available at prediction time. Safe historical feature. |
| avg_position | Keep | Trailing 90-day metric available at prediction time. Safe historical feature. |
| days_since_last_update | Keep | Historical property of the content available at prediction time. Safe historical feature. |

The final model contains no future-window information or label-derived features, relying strictly on trailing observation window metrics.

## 4. Error analysis

*Show top 5 false positives and false negatives, followed by patterns observed.*

In [2]:
df_test = df_test_time.copy()
df_test["lr_prob"] = probs_time
df_test["lr_prediction"] = (df_test["lr_prob"] > 0.5).astype(int)

# False Positives
fps = df_test[(df_test["lr_prediction"] == 1) & (df_test["is_declining"] == 0)]
top_fps = fps.sort_values(by="lr_prob", ascending=False).head(5)

print("Top 5 False Positives:")
for _, row in top_fps.iterrows():
    print(f"Content: {row['content_id']} | Prob: {row['lr_prob']:.3f} | Stale: {row['days_since_last_update']}d | Impr: {row['impressions_90d']} | Pos: {row['avg_position']}")

# False Negatives
fns = df_test[(df_test["lr_prediction"] == 0) & (df_test["is_declining"] == 1)]
top_fns = fns.sort_values(by="lr_prob", ascending=True).head(5)

print("\nTop 5 False Negatives:")
for _, row in top_fns.iterrows():
    print(f"Content: {row['content_id']} | Prob: {row['lr_prob']:.3f} | Stale: {row['days_since_last_update']}d | Impr: {row['impressions_90d']} | Pos: {row['avg_position']}")


Top 5 False Positives:
Content: content_3e79eaafc89d | Prob: 0.908 | Stale: 20d | Impr: 8779 | Pos: 11.6
Content: content_26d48a980581 | Prob: 0.885 | Stale: 106d | Impr: 1266 | Pos: 4.6
Content: content_1452e7edb0a5 | Prob: 0.880 | Stale: 20d | Impr: 2815 | Pos: 10.8
Content: content_1d2233dc3323 | Prob: 0.875 | Stale: 8d | Impr: 1463 | Pos: 1.5
Content: content_db1cd41b4b4f | Prob: 0.869 | Stale: 105d | Impr: 1482 | Pos: 12.9

Top 5 False Negatives:
Content: content_28b4223f4e5f | Prob: 0.012 | Stale: 1d | Impr: 1 | Pos: 0.0
Content: content_a638a00cdb70 | Prob: 0.179 | Stale: 8d | Impr: 3 | Pos: 77.0
Content: content_a29c5d0e1d13 | Prob: 0.238 | Stale: 20d | Impr: 6 | Pos: 67.5
Content: content_fbad54d7a5dc | Prob: 0.247 | Stale: 20d | Impr: 14 | Pos: 79.9
Content: content_f6b05f1a673a | Prob: 0.252 | Stale: 8d | Impr: 7 | Pos: 68.0


**Error Patterns:**
* In these examples, several false positives actually feature relatively fresh pages (8 to 20 days stale) but have very high historical impressions (1,463 to 8,779) and strong average positions (top 15). This suggests the model may over-index on historical visibility metrics to predict decline rather than relying solely on staleness.
* Two of the false positives are indeed older (105+ days), which might align with the model expecting a decay that hasn't happened yet.
* For the false negatives, several involve extremely low impressions (under 15) and poor positions (mostly 67+ or 0.0). One possible explanation is that very low-traffic pages are mathematically easier to trigger a percentage-based "down" trend with just a tiny absolute drop, which the model misses.
* The false negatives shown are also relatively fresh (1 to 20 days). This suggests that recent updates do not guarantee stability, perhaps because these numeric columns cannot capture nuances in the actual textual quality of the update.

## 5. Rewrite my claims

**Original claim**
Our model proves that updating content guarantees an optimal and immediate recovery in search rankings and traffic.

**Rewritten claim**
Our model provides decision-support by identifying content that exhibits directional decline, where observed staleness and measured position data suggest a refresh may be beneficial.